# Implementasi Fine-Tuning Qwen 2.5 dengan Unsloth (QLoRA)

In [ ]:
import torch
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU: {gpu_stats.name} ({max_memory} GB)")
print(f"Initial VRAM Reserved: {start_gpu_memory} GB")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

model_id = "Qwen/Qwen2.5-3B-Instruct"


# Konfigurasi 4-bit NF4 Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True,
    padding_side="right"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load Base Model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Persiapkan model untuk k-bit training (freeze base weights + enable gradient checkpointing)
model = prepare_model_for_kbit_training(model)
print(" Base Model berhasil dimuat dalam 4-bit NF4!")

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=8,
    lora_alpha=64,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


In [ ]:
from datasets import load_dataset
# Sel 5: Load JSONL dataset
from datasets import load_dataset

dataset_path = "data/credit_finetune_dataset.jsonl"
dataset = load_dataset("json", data_files=dataset_path, split="train")

# Sel 6: Fungsi format prompt ChatML
prompt_template = """<|im_start|>system
{}<|im_end|>
<|im_start|>user
{}<|im_end|>
<|im_start|>assistant
{}<|im_end|>"""

EOS_TOKEN = tokenizer.eos_token if tokenizer.eos_token else "<|im_end|>"

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = prompt_template.format(instruction, input_text, output)
        texts.append(text)
    return {"text": texts}

formatted_dataset = dataset.map(formatting_prompts_func, batched=True)
print("Contoh formatted prompt:\n", formatted_dataset[0]["text"][:500])


In [ ]:
# Sel 7: Setup SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="outputs/credit_qwen_qlora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    save_strategy="epoch",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=training_args,
)

# Sel 8: Eksekusi Training & Monitor Loss
trainer_stats = trainer.train()
print(f"Training selesai! Final Loss: {trainer_stats.training_loss:.4f}")


In [ ]:
# Sel 9: Setup inference mode & uji coba output JSON
import json

model.eval()

sample_instruction = "Anda adalah Senior Credit Risk Underwriter AI di institusi perbankan. Tugas Anda adalah mengevaluasi aplikasi kredit pemohon berdasarkan data demografi, keuangan, hasil prediksi model XGBoost (Probability of Default), dan kontribusi faktor risiko matematis (SHAP Values).\n\nHasilkan laporan analisis kredit (Credit Underwriting Memo) yang terstruktur strictly dalam format JSON valid."

sample_input = """### PROFIL PEMOHON PINJAMAN:
- Usia Pemohon: 24 tahun
- Pendapatan Tahunan: $45,000
- Status Kepemilikan Rumah: RENT
- Lama Bekerja: 2.0 tahun
- Tujuan Pinjaman: MEDICAL
- Peringkat Risiko Kredit (Grade): C
- Besaran Pinjaman yang Diajukan: $12,000
- Suku Bunga Pinjaman: 13.50%
- Rasio Pinjaman / Pendapatan: 26.7%
- Riwayat Gagal Bayar Sebelumnya: N
- Panjang Riwayat Kredit: 3 tahun

### KALKULASI RISIKO ML & ANALISIS SHAP:
- Prediksi Probability of Default (PD): 31.2%
- Kategori Risiko Awal: MEDIUM_RISK
- Rekomendasi Awal: MANUAL_REVIEW
- Faktor Pendorong Risiko Terbesar (+SHAP):
  * person_home_ownership_RENT bernilai 1.0 (SHAP: +0.28)
  * Rasio pinjaman terhadap pendapatan sebesar 26.7% (SHAP: +0.22)
- Faktor Pereda Risiko Terbesar (-SHAP):
  * Riwayat gagal bayar bersih (cb_person_default_on_file = N) (SHAP: -0.65)
  * Pendapatan tahunan sebesar $45,000 (SHAP: -0.35)"""

inference_prompt = prompt_template.format(sample_instruction, sample_input, "")
inputs = tokenizer([inference_prompt], return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.2,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id
    )

response_text = tokenizer.batch_decode(outputs, skip_special_tokens=False)[0]
generated_response = response_text.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()

print("Hasil Respon Model:\n", generated_response)

# Validasi JSON Parser
try:
    parsed_json = json.loads(generated_response)
    print("\nJSON VALID! Rekomendasi:", parsed_json.get("recommendation"))
except json.JSONDecodeError as e:
    print("\n JSON Parsing Gagal:", str(e))


In [ ]:
# Sel 10: Simpan LoRA Adapter
adapter_dir = "models/lora_adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"LoRA Adapter tersimpan di: {adapter_dir}")

# Sel 11: Merge LoRA Adapter ke Base Model (Float16) untuk Export GGUF
from peft import PeftModel

base_model_reload = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

merged_model = PeftModel.from_pretrained(base_model_reload, adapter_dir)
merged_model = merged_model.merge_and_unload()

merged_dir = "models/merged_model"
merged_model.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)
print(f" Merged Model tersimpan di: {merged_dir}")
